In [1]:

import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# sys.path.append('/Users/yiqin/Documents/PythonProjects/GraspDataProcessing/src')
# sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')
import graspdataprocessing as gdp


In [2]:
config_file = 'config.toml'  # 配置文件的路径
config = gdp.load_config(config_file)

配置验证通过: cutoff_value=1e-09, initial_ratio=0.09


In [4]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = gdp.setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

gdp.setup_directories(config)
# 初始化结果文件
# gdp.initialize_results_file(config, logger)

# 验证初始文件
gdp.validate_initial_files(config, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-12 13:07:30,270 - INFO - 机器学习训练程序启动
2025-06-12 13:07:30,271 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-12 13:07:30,271 - INFO - 初始比例: 0.09
2025-06-12 13:07:30,273 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [5]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, target_pool_csfs_data, raw_csfs_descriptors, cal_csfs_data, caled_csfs_indices_dict = gdp.load_data_files(config, logger)
    
    # 检查组态耦合
    cal_result = gdp.check_configuration_coupling(config, energy_level_data_pd, logger)
    logger.info("************************************************")

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

2025-06-12 13:07:39,090 - INFO - 加载能级数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 332.96it/s]
2025-06-12 13:07:39,102 - INFO - 加载 *.m 文件数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m


cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


2025-06-12 13:07:45,212 - INFO - 加载初始 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4
2025-06-12 13:07:45,783 - INFO - 加载初始 CSFs 描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4


Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors_block_indices.npy
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-12 13:07:47,000 - INFO - 加载本轮计算 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c
2025-06-12 13:07:47,013 - INFO - 加载本轮选择的 CSFs 的索引文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_chosen_indices.pkl
2025-06-12 13:07:47,015 - INFO - cal_loop 1 组态耦合正确
2025-06-12 13:07:47,015 - INFO - ************************************************


In [6]:
caled_csfs_descriptors = gdp.generate_chosen_csfs_descriptors(config, caled_csfs_indices_dict, raw_csfs_descriptors, rmix_file_data, logger)



2025-06-12 13:07:59,803 - INFO - 保存本轮选择的 CSFs 的描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.npy


Descriptors saved to: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_descriptors.npy


In [7]:
model, X_train, X_test, y_train, y_test, training_time, weight = gdp.train_model(config, caled_csfs_descriptors, rmix_file_data, logger)

2025-06-12 13:08:01,393 - INFO - 训练集 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-12 13:08:03,015 - INFO - 创建新模型，设置类别权重: 负样本=1.0, 正样本=13.0
2025-06-12 13:08:03,017 - INFO - 使用原始数据训练 - 不进行重采样
2025-06-12 13:08:03,017 - INFO - 最终训练数据 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-12 13:08:03,017 - INFO - 使用类别权重和损失函数来处理数据不平衡问题
2025-06-12 13:08:03,018 - INFO -              训练模型
2025-06-12 13:08:03,018 - INFO - 开始训练 - 数据量:308,480, 特征维度:87
2025-06-12 13:08:18,666 - INFO - Epoch [50/150], Loss: 0.328169
2025-06-12 13:08:33,605 - INFO - Epoch [100/150], Loss: 0.316850
2025-06-12 13:08:48,515 - INFO - Epoch [150/150], Loss: 0.308164
2025-06-12 13:08:48,516 - INFO - 训练Loss良好 (0.310)，模型收敛效果理想
2025-06-12 13:08:48,516 - INFO -              预测与评估
2025-06-12 13:08:48,618 - INFO - 预测概率统计 - 最小值:0.0000, 最大值:1.0000, 平均值:0.2616
2025-06-12 13:08:48,618 - INFO - 预测为正类的样本数: 5657/77120
2025-06-12 13:08:48,619 - INFO - 真实正样本数: 5423.0/77120
2025-06-12 13:08:48,619 - INFO - 真实正样本比例: 0.070, 预测正样本比例: 0.073


(385600,)


2025-06-12 13:08:50,232 - INFO - 测试集预测结果:
2025-06-12 13:08:50,232 - INFO - AUC:0.8804529177176255, pr_auc:0.765491967324128, f1:0.7454873646209387, accuracy:0.9634336099585062, precision:0.7300689411348772, recall:0.7615710861146967
2025-06-12 13:08:50,339 - INFO - 训练集预测结果:
2025-06-12 13:08:50,340 - INFO - AUC:0.945704377369752, f1:0.7645742919779875, accuracy:0.965884336099585, precision:0.7540151782562654, recall:0.775433342408567


In [9]:
stay_csfs_descriptors = gdp.get_stay_descriptors(raw_csfs_descriptors, caled_csfs_indices_dict)
X_stay = stay_csfs_descriptors.copy()

In [10]:
evaluation_results = gdp.evaluate_model(
            model, X_train, X_test, y_train, y_test, X_stay, config, logger
        )

2025-06-11 11:27:43,644 - INFO - 开始预测与评估
2025-06-11 11:27:44,700 - INFO - 模型评估完成


In [ ]:
# 访问结果
test_predictions = evaluation_results['predictions']['y_pred_test']
test_probabilities = evaluation_results['probabilities']['y_proba_test']
test_f1 = evaluation_results['test_metrics']['f1']
train_f1 = evaluation_results['train_metrics']['f1']

In [ ]:
overfitting_check = test_f1 - train_f1  # 如果差异过大说明过拟合

In [17]:
logger.info("             模型推理")
start_time = time.time()
X_stay = stay_csfs_descriptors.copy()
y_stay_pred = model.predict(X_stay)
y_stay_proba = model.predict_proba(X_stay)[:, 1]
eval_time = time.time() - start_time
logger.info(f"             模型推理时间:{eval_time}")

2025-06-11 10:15:38,306 - INFO -              模型推理
2025-06-11 10:15:39,164 - INFO -              模型推理时间:0.8577296733856201


In [13]:
y_pred = evaluation_results['predictions']['y_pred_test']
y_proba = evaluation_results['probabilities']['y_proba_test']
result_file_path = config.root_path / 'test_data' / f'{config.conf}_{config.cal_loop_num}.csv'
pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "y_proba": y_proba}).to_csv(result_file_path, index=False)

In [29]:
y_pred

array([False, False, False, ..., False, False, False])

In [30]:
np.where(y_pred == 1)[0]

array([    9,    23,    35, ..., 77080, 77111, 77115])

In [24]:
y_stay_pred = model.predict(X_stay)

In [40]:
ml_predict_index = np.where(y_stay_pred == 1)[0]

In [32]:
y_stay_proba = model.predict_proba(X_stay)[:, 1]

In [34]:
y_stay_proba.shape

(3898846,)

In [36]:
y_stay_pred.shape

(3898846,)

In [37]:
target_pool_csfs_data.CSFs_block_length

array([4284446])

In [38]:
cal_csfs_data.CSFs_block_length

array([385600])

In [41]:
ml_predict_index.shape

(30679,)

In [42]:
rmix_file_data.mix_coefficient_List

[array([[ 3.88368822e-01, -3.75787854e-01,  2.86599207e-01, ...,
          1.36427027e-07,  6.39849076e-07,  5.39727752e-08],
        [ 1.24814669e-02, -1.21756193e-02,  9.17219688e-03, ...,
         -6.31284283e-07,  5.70214263e-07,  3.64236044e-07]])]

In [45]:
mix_coeff_above_threshold = gdp.batch_asfs_mix_square_above_threshold(rmix_file_data, threshold=config.cutoff_value)

In [49]:
mix_coeff_above_threshold[0].shape

(26971,)

In [50]:
selected_indices_file = Path(config.root_path) / f"{config.conf}_selected_indices.pkl"
selected_csfs_indices_dict = gdp.csfs_index_load(selected_indices_file)

In [47]:
llasdg = np.union1d(ml_predict_index, mix_coeff_above_threshold[0])

In [48]:
llasdg.shape

(56594,)

In [54]:
energy_level_data_pd['EnergyTotal'][0]

np.float64(-11274.7413433)